# Lab 02: From notebook to a command you can run again

Automated Workflows in Satellite Image Processing · v1.0 · 5 October 2026

Your Lab 01 service discovers real Sentinel scenes and stores its progress. Today you turn it into a program you can run from a terminal whenever your laptop is on.

Use this notebook to predict behavior, run checks and explain results. Edit **student_tasks.py** in a text editor. Run the actual service from a terminal. The service never depends on a notebook kernel.

By the end you can run the same command several times over a week, inspect its history, and explain whether it succeeded even when it found zero new scenes.

Suggested lab time: 90 minutes, followed by four short runs over seven days. No GPU, cluster or continuously running laptop is needed.


## What comes from Lab 01?

`lab01_service.py` contains the working schema, database insertion, pagination, overlapping discovery, and safe one-JPEG downloader extracted from Lab 01 instructor v3. It has no notebook execution side effects.

`workflow.py` supplies the orchestration and two small additions: a run journal and a link between each query stream and its scenes. The link prevents selecting a preview from a different region when a database contains several streams.

`sentinel_cli.py` supplies the argument parser. Your three repairs go in `student_tasks.py`. The captured provider pages and real preview in `fixtures/` are reused from Lab 01 only for repeatable checks. Operational discovery always uses live HTTPS requests; it never substitutes recorded data after a network failure.

Do not edit the supplied service or add a timer loop.


## Setup

Extract the student ZIP. Open a terminal in the extracted folder containing `sentinel_cli.py`, using your existing Python environment. If you need the notebook interface:

```bash
python -m pip install -r requirements.txt
```

```bash
python -m jupyterlab Lab02_Sentinel_CLI_v1_20261005.ipynb
```

The command line service and checks themselves need no third party packages. Keep the folder together. If you launch Jupyter from somewhere else, set `PROJECT` below to this folder's absolute path.


In [1]:
from pathlib import Path
import json
import sqlite3
import subprocess
import sys

PROJECT = Path.cwd().resolve()
assert (PROJECT / 'sentinel_cli.py').is_file(), 'Set PROJECT to the extracted student folder'

def run_check(exercise=None):
    args = [sys.executable, str(PROJECT / 'check_lab02.py')]
    if exercise is not None:
        args += ['--exercise', str(exercise)]
    result = subprocess.run(args, cwd=PROJECT, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print('Exit code:', result.returncode)
    return result.returncode

print('Project:', PROJECT)


Project: D:\agh\Automated_Workflows\Labs\Lab02


## Before editing: inspect the command interface (5 minutes)

Run these in your terminal:

```bash
python sentinel_cli.py --help
```

```bash
python sentinel_cli.py discover --help
```

Read `config/krakow_week.json`. Where is the region specified? What makes this a repeatable query? Why is there an initial start but no permanent end date?

There are deliberate bugs in the three task functions. The historical demonstration end means the starter is not ready for the weekly experiment.


## Exercise 1: One configuration, one persistent state (20 minutes)

Read `resolve_paths` in `student_tasks.py`. Imagine running the program first from its own folder and then from its parent. Predict which database each invocation opens.

Repair the function so relative database, preview and log paths are anchored to the configuration file's parent. Keep absolute paths absolute. Resolve paths without creating files or changing the process working directory.

This matters because the scheduler or terminal decides the working directory. A successful command can silently create a second, empty catalogue if path handling is wrong.

Run the check before and after the repair:


In [3]:
run_check(1)


PASS: 1

Exit code: 0


0

Write your explanation here: why is anchoring these paths to the configuration file more reliable than anchoring them to the terminal folder? What would happen if you deliberately changed the database path?

Anchoring these paths to the configuration file is more reliable tahn to the terminal folder because it ensures that the application locates its database, logs and download directory refardless of where the command is used from. Changing the database path would case the application to connect to empty databse at the new location and it would lose track of stored data.

## Exercise 2: Discover through the current time (20 minutes)

Read `choose_until`. The prototype copied Lab 01's demonstration end. That was useful for a fixed experiment, but it cannot discover this week's acquisitions.

Repair it so `--until` takes priority when supplied; otherwise capture an aware UTC timestamp once per invocation. Normalize explicit timestamps to UTC and reject values without a timezone. The optional `clock` argument lets a check control time without changing the satellite records.

Do not set the search start to “now minus one day.” The supplied discovery service uses the saved watermark minus the overlap, bounded by the original initial start.


In [5]:
run_check(2)


PASS: 2

Exit code: 0


0

Write your explanation here: if you run on Monday and next on Thursday, which part of the configuration and saved state determines the next search interval? Why must the upper bound remain fixed while following pages?

The start of the interval is determined by the watermark adjusted by the overlap window and bounded by the start date. The until is determined by the current UTC clock. If the upper bound would not remain fixed while following pages then the pages could introduce duplicates or missed records and it would break the pagination.

## Exercise 3: Make the terminal tell the truth (20 minutes)

Read `execute_job`. Predict what a caller sees when the network request fails. Repair the function so success returns exit code 0 and the actual result; failure logs a useful error and returns exit code 1 with `status='failed'` and an error description.

A completed search with zero new scenes is a successful run. A connection error is a failed run. Do not infer success from a positive scene count.

The check also launches the actual CLI with a missing configuration file to verify that the exit code reaches the terminal.


In [6]:
run_check(3)


PASS: 3

Exit code: 0


0

After your repair, try this deliberate failure in the terminal:

```bash
python sentinel_cli.py discover --config config/missing.json
```

Inspect the exit code immediately afterward. In Git Bash or Bash:

```bash
echo $?
```

In PowerShell:

```powershell
$LASTEXITCODE
```

Explain what a future scheduler could do with this code. A missing configuration cannot provide a log path, so this early error appears on the terminal; operational errors also go to the configured log.

Future scheduler could automate monitoring and trigger alert notifications or retry the program.

## Exercise 4: Verify repeated execution (15 minutes)

Run every check. In addition to your three repairs, these reuse actual captured Lab 01 records to verify interrupted discovery, recovery, repeat sightings, and one verified preview. Only clocks and transport failures are controlled.


In [7]:
run_check()


PASS: 1
PASS: 2
PASS: 3
PASS: service recovery
PASS: one real preview

Exit code: 0


0

Now run the live program. Run one command at a time; simultaneous writers are outside this lab's scope. The first command discovers metadata and requests at most one preview. Preview download is optional on subsequent runs.

```bash
python sentinel_cli.py discover --config config/krakow_week.json --preview
```

```bash
python sentinel_cli.py discover --config config/krakow_week.json
```

```bash
python sentinel_cli.py report --config config/krakow_week.json
```

To demonstrate stable paths, move to the parent of the extracted folder and invoke the same program/configuration with their paths. Adapt the folder name to your extraction:

```bash
python Lab02_Sentinel_CLI_v1_20261005/sentinel_cli.py report --config Lab02_Sentinel_CLI_v1_20261005/config/krakow_week.json
```

Compare the reported absolute database path with the earlier report. It must be identical.

Live availability is not guaranteed. If a request fails, keep the log, inspect the saved watermark and retry later. Do not treat an offline check as evidence of a successful live run.


## Inspect your actual state

This cell only reads the current project's database. It does not perform discovery. If you changed the configured database path, update `DB` accordingly.


In [8]:
DB = (PROJECT / 'state' / 'catalogue.sqlite').resolve()
if not DB.is_file():
    print('Run live discovery from the terminal first; no database is created here.')
else:
    with sqlite3.connect(f'{DB.as_uri()}?mode=ro', uri=True) as conn:
        conn.row_factory = sqlite3.Row
        runs = [dict(row) for row in conn.execute(
            'SELECT started_at, status, search_start, search_until, pages, sightings, new_scenes, latest_acquired, error FROM runs ORDER BY started_at, rowid')]
        print(json.dumps(runs, indent=2))
        print('Saved query watermarks:')
        for row in conn.execute('SELECT stream_key, watermark FROM streams'):
            print(dict(row))


[
  {
    "started_at": "2026-10-12T17:17:33.010730Z",
    "status": "succeeded",
    "search_start": "2026-09-21T00:00:00Z",
    "search_until": "2026-10-12T17:17:32.983474Z",
    "pages": 3,
    "sightings": 24,
    "new_scenes": 24,
    "latest_acquired": "2026-10-10T09:56:57.028000Z",
    "error": null
  },
  {
    "started_at": "2026-10-12T17:18:32.962943Z",
    "status": "succeeded",
    "search_start": "2026-10-05T17:17:32.983474Z",
    "search_until": "2026-10-12T17:18:32.960752Z",
    "pages": 2,
    "sightings": 8,
    "new_scenes": 0,
    "latest_acquired": "2026-10-10T09:56:57.028000Z",
    "error": null
  }
]
Saved query watermarks:
{'stream_key': '9ae6b6299dd613ee619b8bb36c584f70051bb153fd582d37bc644c3ca29a0420', 'watermark': '2026-10-12T17:18:32.960752Z'}


Explain your two consecutive runs. Which number counts sightings, and which counts scene identities first associated with this stream? Why may the second run have many sightings and zero new scenes? If new imagery arrived between runs, explain the difference rather than requiring a zero count.

If discovery fails after page 1, committed scenes remain and the discovery watermark does not advance. If discovery completes but its optional preview fails, the command fails while the discovery watermark has already advanced. These outcomes are recorded separately in the run journal and download status. Retry the same preview command to recover.

Run 1 is the initial baseline run, it searched from the start date up to the current time and it paged through 3 pages, encountered 24 sightings and they were all new. Run 2 was an immediate follow-up run and due to overlap window it started earlier and paged 2 pages resulting in 8 slightings but they had been already discovered. If new imagery arrieved between runs slightings would count all items and new_scenes would be greater than zero and would tell how many new arrived items were added.

## The seven day experiment

Run discovery at least four times over seven days whenever your laptop is on and online. Keep the same region, initial start and database throughout. Include one immediate rerun and one gap of at least 48 hours. A short run needs no permanent background process.

```bash
python sentinel_cli.py discover --config config/krakow_week.json --preview
```

You may omit `--preview` for metadata-only runs. It always selects the newest available suitable JPEG in this stream, without a cloud filter; a very cloudy preview is still a valid acquisition. It may reuse the same verified JPEG on several days. A JPEG is not a scientific raster product or a quality assessment.

At the end, save a report:

```bash
python sentinel_cli.py report --config config/krakow_week.json > week_report.json
```

Complete this table from the actual run history:

| Run time (UTC) | Outcome | Search interval | Sightings | Newly discovered scenes | Newest acquisition | Observation |
|---|---|---|---|---|---|---|
|2026-10-12 17:17:33 |succeeded |2026-09-21T00:00:00Z / 2026-10-12T17:17:32Z|24 |24 |2026-10-10 09:56:57Z|First run download the entire history and got all 24 new scenes |
|---|---|---|---|---|---|---|
|2026-10-12 17:18:32 |succeeded | 2026-10-05T17:17:32Z / 2026-10-12T17:18:32Z|8 |0 |2026-10-10 09:56:57Z |The next run did not find any new scenes |
|---|---|---|---|---|---|---|
|2026-10-14 20:16:41 |succeeded |2026-10-05T17:44:54Z / 2026-10-14T20:16:41Z |10 |2 |2026-10-13 09:56:43Z |Run after 48 h got 2 new scenes with new acquisition date |
|---|---|---|---|---|---|---|
|2026-10-17 14:16:33 |succeeded |2026-10-07T20:16:41Z / 2026-10-17T14:16:33Z |10 |2 |2026-10-15 09:46:48Z |Another 2 new scenes and new acquisition date |
|---|---|---|---|---|---|---|
|2026-10-18 13:24:51 |succeeded |2026-10-10T14:16:33Z / 2026-10-18T13:24:51Z |6 |2 |2026-10-17 09:46:57Z |After all runs a total of 30 unique scenes were collected in the database |
|---|---|---|---|---|---|---|

Discuss: did skipping days lose scenes within the covered search interval? How old was the newest acquisition? Did a new discovery necessarily mean a recent acquisition? What evidence distinguishes “no new scenes” from “the request failed”? Was the newest preview also the clearest?

Skipping days did not cause any data loss, because of the overlap period. The newest acquisition was over one day old. New discovery does not have to mean a recent acquisition it only means that a scene was added to the database during a run. "No new scenes" is a successful execution with no new senes, "the request failed" indicates that there was an error. The newest preview was the clearest, but it is used for quick visual inspection rather than analysis.

This is polling, not an immediate notification service. Acquisition time is not catalogue publication time. Finite overlap can miss items published very late with older acquisition dates; a complete search also cannot prove that the provider has every acquisition. Try a wider lookback as a reconciliation exercise:

```bash
python sentinel_cli.py discover --config config/krakow_week.json --lookback-days 30
```

This cannot go before `initial_start`. Set the initial start once before your first operational run; do not move it forward every day. The shipped start is 21 September 2026, two weeks before this edition. For a future teaching date, update it before starting the experiment.


## Submission

Submit your repaired `student_tasks.py`, this notebook with checks and explanations, `week_report.json`, and your completed observation table. Include a short relevant log excerpt if a live run failed. Do not commit preview images, database files or the entire log to Git.

Your service is now ready to be run by another program. Containers and cluster scheduling can be added in later labs without moving the operational logic back into the notebook.

Provider documentation: [Element 84 Earth Search](https://github.com/Element84/earth-search). Lab 01 source: `Lab01_Sentinel_RealData_Repair_v3_20261005_Instructor.ipynb`. Recorded fixtures retain the original provider records and real JPEG.
